In [1]:
import pandas as pd
import numpy as np
import geopandas as gp
import h3
import json
import folium
from shapely.geometry import shape
import matplotlib

## 1. Overview

This notebook transforms the cleaned dataset produced in `01_eda_bogota.ipynb` into a model-ready feature matrix. The main steps are:

1. **H3 spatial indexing** — derive a cell index from `lat`/`lon` to encode location
2. **IDECA neighborhood join** — assign each listing to an official Bogotá neighborhood polygon
3. **Categorical encoding** — encode `tipo_propiedad`, `estrato`, and spatial features
4. **Target transformation** — apply `log` to `precio`
5. **Train / validation / test split**

In [4]:
df = pd.read_csv('data/feature_engineering.csv') 

## 2. H3 Spatial Indexing

H3 (Uber's hexagonal hierarchical spatial index) partitions the map into hexagonal cells at a given resolution. Each listing is assigned to a cell based on its `lat`/`lon` coordinates.

We use **resolution 8** (~460m cell edge) as a balance between spatial granularity and minimum samples per cell. Resolution 9 (~175m) was considered but risks sparse cells in low-density areas of the city.

In [5]:
H3_RESOLUTION = 8

df['h3_index'] = df.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], H3_RESOLUTION), axis=1
)

In [6]:
# Distribution of listings per H3 cell
cell_counts = df['h3_index'].value_counts()
cell_counts.describe()

count     682.000000
mean       86.473607
std       146.626620
min         1.000000
25%         3.000000
50%        33.000000
75%        94.000000
max      1090.000000
Name: count, dtype: float64

In [7]:
df_agg = df.groupby('h3_index')['precio'].agg(['min','max','median','count']).reset_index()
df_agg

,h3_index,min,max,median,count
0,88089b1a63fffff,2270000000,2270000000,2.270000e+09,1
1,8827592027fffff,200000000,200000000,2.000000e+08,2
2,8839017111fffff,1770000000,2500000000,2.175000e+09,4
3,88660b4835fffff,330000000,330000000,3.300000e+08,1
4,88660b8647fffff,1380000000,1380000000,1.380000e+09,1
...,...,...,...,...,...
677,88754a9a59fffff,225000000,225000000,2.250000e+08,1
678,88754e6499fffff,80000000,5000000000,4.225000e+08,308
679,88754e64d9fffff,350000000,350000000,3.500000e+08,1
680,88a94e5557fffff,290000000,290000000,2.900000e+08,1


In [8]:
from geojson import Feature, Point, FeatureCollection

In [3]:
def hexagons_dataframe_to_geojson(df_hex, file_output = None, column_name = "value"):
    """
    Produce the GeoJSON for a dataframe, constructing the geometry from the "hex_id" column
    and with a property matching the one in column_name
    """    
    list_features = []
    
    for i,row in df_hex.iterrows():
        try:
            geometry_for_row = { "type" : "Polygon", "coordinates": [[[lng, lat] for lat, lng in h3.cell_to_boundary(row["h3_index"])]]}
            feature = Feature(geometry = geometry_for_row , id=row["h3_index"], properties = {column_name : row[column_name]})
            list_features.append(feature)
        except Exception as e:
            print("An exception occurred for hex " + row["h3_index"]) 
            raise e

    feat_collection = FeatureCollection(list_features)
    geojson_result = json.dumps(feat_collection)
    return geojson_result

In [2]:
def get_color(custom_cm, val, vmin, vmax):
    return matplotlib.colors.to_hex(custom_cm((val-vmin)/(vmax-vmin)))

In [ ]:
def choropleth_map(df_aggreg, column_name = "value", border_color = 'black', fill_opacity = 0.7, color_map_name = "Blues", initial_map = None):
    """
    Creates choropleth maps given the aggregated data. initial_map can be an existing map to draw on top of.
    """    
    #colormap
    min_value = df_aggreg[column_name].min()
    max_value = df_aggreg[column_name].max()
    mean_value = df_aggreg[column_name].mean()
    print(f"Colour column min value {min_value}, max value {max_value}, mean value {mean_value}")
    print(f"Hexagon cell count: {df_aggreg['hex_id'].nunique()}")
    
    # the name of the layer just needs to be unique, put something silly there for now:
    name_layer = "Choropleth " + str(df_aggreg)
    
    if initial_map is None:
        initial_map = folium.Map(location= [47, 4], zoom_start=5.5, tiles="cartodbpositron")

    #create geojson data from dataframe
    geojson_data = hexagons_dataframe_to_geojson(df_hex = df_aggreg, column_name = column_name)

    # color_map_name 'Blues' for now, many more at https://matplotlib.org/stable/tutorials/colors/colormaps.html to choose from!
    custom_cm = matplotlib.cm.get_cmap(color_map_name)

    folium.GeoJson(
        geojson_data,
        style_function=lambda feature: {
            'fillColor': get_color(custom_cm, feature['properties'][column_name], vmin=min_value, vmax=max_value),
            'color': border_color,
            'weight': 1,
            'fillOpacity': fill_opacity 
        }, 
        name = name_layer
    ).add_to(initial_map)

    return initial_map

In [10]:
geojson = hexagons_dataframe_to_geojson(df_hex=df_agg,column_name='h3_index')

In [9]:
initial_map = folium.Map(location=[4.711, -74.0721], zoom_start=11, tiles="cartodbpositron")

In [11]:
folium.GeoJson(
    geojson,
    name = 'prueba'
).add_to(initial_map)

In [12]:
initial_map

## 3. IDECA Neighborhood Join

IDECA (Infraestructura de Datos Espaciales para el Distrito Capital) provides the official neighborhood boundary polygons for Bogotá. Each listing is spatially joined to its corresponding neighborhood polygon using its projected coordinates (EPSG:3116).

This feature captures fine-grained location quality that complements the H3 index — two listings in the same H3 cell may belong to different neighborhoods with distinct price profiles.

In [ ]:
# Load IDECA neighborhood polygons
# ideca = gp.read_file('data/SECTOR.geojson').to_crs(epsg=3116)

# Spatial join — assign neighborhood name to each listing
# df_geo = gp.GeoDataFrame(df, geometry=gp.points_from_xy(df['lat'], df['lon']), crs='EPSG:4326').to_crs(epsg=3116)
# df = gp.sjoin(df_geo, ideca[['SCANOMBRE', 'geometry']], how='left', predicate='within')
# df = df.rename(columns={'SCANOMBRE': 'barrio_ideca'}).drop(columns=['geometry', 'index_right'])

## 4. Categorical Encoding

Tree-based models (LightGBM) can handle categorical features natively, but require them to be encoded as integers or pandas `Categorical` dtype.

- **`tipo_propiedad`** — binary: `apartamento=0`, `casa=1`
- **`estrato`** — ordinal 1–6, already numeric after imputation
- **`h3_index`** — label encoded (high cardinality, LightGBM handles via `categorical_feature`)
- **`barrio_ideca`** — label encoded (high cardinality)

In [ ]:
from sklearn.preprocessing import LabelEncoder

df['tipo_propiedad'] = df['tipo_propiedad'].map({'apartamento': 0, 'casa': 1})
df['estrato'] = df['estrato'].astype(int)

le_h3 = LabelEncoder()
df['h3_index_enc'] = le_h3.fit_transform(df['h3_index'])

# le_barrio = LabelEncoder()
# df['barrio_enc'] = le_barrio.fit_transform(df['barrio_ideca'].fillna('unknown'))

## 5. Target Transformation

As established in the EDA (section 9.4), the `precio` column is log-transformed to produce a near-normal distribution. The model is trained on `log(precio)` and predictions are converted back via `exp` during inference.

In [ ]:
df['log_precio'] = np.log(df['precio'])

## 6. Final Feature Set

| Feature | Type | Notes |
|---|---|---|
| `area_m2` | numeric | bounded 18–5,000 m² |
| `cuartos` | numeric | imputed via area buckets |
| `banios` | numeric | imputed via estrato + tipo |
| `parqueaderos` | numeric | imputed via tipo + area_bin |
| `tipo_propiedad` | binary | 0=apartamento, 1=casa |
| `estrato` | ordinal | 1–6, spatially imputed |
| `h3_index_enc` | categorical | resolution 8 |
| `barrio_ideca` | categorical | IDECA polygons |
| `log_precio` | numeric | target variable |

## 7. Train / Validation / Test Split

The dataset is split into three sets:

- **Train (70%)** — used to fit the model
- **Validation (15%)** — used for hyperparameter tuning and early stopping
- **Test (15%)** — held out, evaluated only once at the end

Stratification by `tipo_propiedad` ensures both classes are proportionally represented in all splits.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ['area_m2', 'cuartos', 'banios', 'parqueaderos', 'tipo_propiedad',
            'estrato', 'h3_index_enc']
TARGET = 'log_precio'

X = df[FEATURES]
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=df['tipo_propiedad']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=X_temp['tipo_propiedad']
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')